### Question 1

Create a report for 2022 monthwise. Expected output should be "month_year", "source", "total_count"


In [0]:
jobs_data = [('Alice','2022-02-01 12:14:46'),('Bob','2022-01-26 22:05:25'),('Charlie','2022-03-25 12:00:00'),('David','2022-05-12 12:00:00'),('Eve','2022-07-15 12:00:00'),('Frank','2022-09-25 07:12:34'),('Grace','2022-11-25 12:00:00'),('Henry','2022-07-25 12:00:00'),('Ivy','2022-12-25 12:00:00'),('Jack','2021-12-25 12:00:00'),('Kate','2022-01-25 12:00:00'),('Liam','2022-05-25 12:00:00'),('Mia','2022-08-11 16:12:34')]

freelancers_data = [('Alice','2022-02-01 12:14:46'),('Bob','2022-01-26 22:05:25'),('Charlie','2022-03-25 12:00:00'),('David','2022-05-12 12:00:00'),('Eve','2022-07-15 12:00:00'),('Frank','2022-09-25 07:12:34'),('Grace','2022-11-25 12:00:00'),('Henry','2022-07-25 12:00:00'),('Ivy','2022-12-25 12:00:00'),('Jack','2021-12-25 12:00:00'),('Kate','2022-01-25 12:00:00'),('Liam','2022-05-25 12:00:00'), ('Mia','2022-08-11 16:12:34'),('Nina','2022-08-11 16:12:34'), ('Oliver','2022-08-11 16:12:34'), ('Pablo','2022-08-11 16:12:34'), ('Quinn','2022-08-11 16:12:34'), ('Ravi','2022-08-11 16:12:34'), ('Sara','2022-08-11 16:12:34'), ('Tom','2022-08-11 16:12:34'), ('Uma','2022-08-11 16:12:34'), ('Victor','2022-08-11 16:12:34'), ('Wendy','2022-08-11 16:12:34'), ('Xavier','2022-08-11 16:12:34'), ('Yara','2022-08-11 16:12:34'), ('Zoe','2022-08-11 16:12:34')]

columns = ['full_name','dt']

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("examples").getOrCreate()

In [0]:
df_jobs = spark.createDataFrame(jobs_data, columns)
df_jobs.createOrReplaceTempView("jobs")

df_freelancers = spark.createDataFrame(freelancers_data, columns)
df_freelancers.createOrReplaceTempView('freelancers')

In [0]:
%sql
select date_format(dt, 'yyyy-MM') as month_year, 
      'Jobs' as Source, count(*) as total_count 
from jobs
where year(dt) = '2022'
group by  month_year

UNION ALL 

select date_format(dt, 'yyyy-MM') as month_year, 
      'Freelancers' as Source, count(*) as total_count 
from freelancers
where year(dt) = '2022'
group by  month_year

order by month_year

##### Tips .. Displaying the dayofWeek for a given date. 


##### date_format()

Converts a **DATE/TIMESTAMP** to a formatted string. Useful for reporting and time-based aggregations.

```text
yyyy  -> 2026      Year
yy    -> 26

MM    -> 09        Month Number
MMM   -> Sep       Month Name
MMMM  -> September

dd    -> 28        Day
E     -> Mon       Day Name
EEEE  -> Monday

HH    -> 14        Hour (24-hour)
hh    -> 02        Hour (12-hour)
mm    -> 30        Minute
ss    -> 45        Second
a     -> PM        AM/PM

######  Example of the date_format

In [0]:
%sql
select *,date_format(dt, 'EEEE') as dayOfWeek 
        ,date_format(dt, 'yyyy-MM') as monthOfTheYear
        ,date_format(dt, 'dd') as dateOfWeek
        ,date_format(dt, 'hh:ss a') as timeIn12HrsFormat
from jobs
limit 5

## Question 2
Implement a program that manages access to a bank vault, allowing authorized employees to perform sepcific activtiies inisde the vault. The system should implement two classes : **VaultAccessControl** andf **BankVaultAccessManager**


**Sample Input:**   
```Text
5                                               # number of employees
EMP1234, EMP5678, EMP9101, EMP2344, EMP6532     # Employee Ids
1                                               # No of autorized employees
EMP9101                                         # Autorized employees
4                                               # No of users trying to access the vault
EMP1234, EMP5678, EMP9101, EMP653 1             # Employees trying to access the vault
```

**Sample Output:** 
```Text
Error occured: Unauthorized access. Employee ID EMP1234 not authorized to access the vault.
Error occured: Unauthorized access. Employee ID EMP5678 not authorized to access the vault.
Authorized Access for EMP9101. Performed activity inside the bank vault: Counting cash
Error occured: Unauthorized access. Employee ID EMP6531 not recognized. 
```

In [0]:
class VaultAccessControl:
    def __init__(self,authorized_employees, all_employees):
        self.authorized_employees = authorized_employees
        self.all_employees = all_employees
        self.locked = False

    def grant_access(self, employee):
        if employee not in self.all_employees:
            raise PermissionError(f"Employee ID {employee} not recognized.")
        if employee not in self.authorized_employees:
            raise PermissionError(f"Employee ID {employee} not authorized to access the vault.")
        self.locked = True
        return True

    def release_access(self):
        self.locked = False

class BankVaultAccessManagement:
    def __init__(self,employee_id, access_control):
        self.employee_id = employee_id
        self.access_control = access_control
        self.access_granted = False

    def __enter__(self):
        try:
            self.access_control.grant_access(self.employee_id)
            self.access_granted = True
            return self
        except PermissionError as e:
            print(f"Error occured: Unauthorized access. {e}")
            return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self.access_granted:
            self.access_control.release_access()

    def perform_activity(self,activity):
        if self.access_granted:
            print(f"Authorized access for {self.employee_id}. Performed activity inside the bank vault: {activity}")

In [0]:
def main(inputData):
    import sys
#    input_data = sys.stdin.read().splitlines()
    input_data = inputData.strip().splitlines()
    if not input_data:
        return
    
    #parse data
    n = int(input_data[0].strip()) 

    # all_employees = [x.strip() for x in input_data[1].split(',') if x.strip()]
    all_employees = []
    for x in input_data[1].split(','):
        if x.strip():
            all_employees.append(x.strip())
    
    m = int(input_data[2].strip())
    
    # authorized_employees = [x.strip() for x in input_data[3].split(',') if x.strip()]
    authorized_employees = []
    for x in input_data[3].split(','):
        if x.strip():
            authorized_employees.append(x.strip())
    
    q = int(input_data[4].strip())
    
    # test_users = [x.strip() for x in input_data[5].split(',') if x.strip()]
    test_users = []
    for x in input_data[5].split(','):
        if x.strip():
            test_users.append(x.strip())
    
    #  Initialize Vault access control
    vault_control = VaultAccessControl(authorized_employees, all_employees)
    
    # process each user attempt
    for user_id in test_users:
        with BankVaultAccessManagement(user_id, vault_control) as manager:
            manager.perform_activity("Counting Cash")

if __name__ == "__main__":
    inputData = """
    5                                               
    EMP1234, EMP5678, EMP9101, EMP2344, EMP6532     
    1                                               
    EMP9101                                         
    4                                               
    EMP1234, EMP5678, EMP9101, EMP6531
    """
    main(inputData)

## Question 3
**PySpark Interview Challenge: Sessionization + SCD Type 2 + Point-in-Time Join**

``` Text
You are a Senior Data Engineer at RepoSphere, a large-scale Source Control Management (SCM) and developer collaboration platform.

Your activity dashboards have outgrown simple daily counts. Product teams now require:

A view of developer work sessions.
A faithful point-in-time view of repository metadata that survives repository renames and visibility changes.

You have two append-only data sources:

1. A stream of per-actor events such as: (Pushes, Pull Requests, Issues) with event_id, event_type, created_at, repo & actor
2. A stream of repository metadata change events - (repo_name, visibility, default_branch) taken at changed_at timestamps. Changes may arrive late (out of order with respect to changed_at).
```
**Build a batch PySpark pipeline that:**

1. Sessionizes per-actor activity (consecutive events with a gap <= 30 min belong to the same session).
2. Builds an SCD Type 2 dimension on repository metadata.
3. Performs a Point-In-Time (PIT) join between events and repository state.
4. Produces a per-session summary.


**Input files:**

```
1. events.json

JSON
{
"event_id": "...",
"event_type": "...",
"created_at": "...",
"repo": {"id": ..., "name": "..."},
"actor": {"id": ..., "login": "..."}
}
```

```
2. repo_changes.json

JSON
{
"change_id": ...,
"repo_id": ...,
"changed_at": "...",
"repo_name": "...",
"visibility": "...",
"default_branch": "..."
}
```
```
The project is partially implemented.

Implement the methods in:

Python
src/main/job/pipeline.py
```
**_`Method 1`_**
**read_events**(self, input_path: str) -> DataFrame

1. Read JSON Lines file.
2. Return DataFrame of nested events.
3. Must not assume all nested fields exist.
```

```
**_`Method 2`_**
**read_repo_changes**(self, input_path: str) -> DataFrame

1. Read JSON Lines file.
2. Return DataFrame of repository metadata changes.
```
DataFrame must expose at least:
change_id
repo_id
changed_at
repo_name
visibility
default_branch
```
```
```
**_`Method 3`_**
**flatten_events**(self, events_df: DataFrame) -> DataFrame

1. Transform raw nested events into an analysis-friendly DataFrame.
2. Parse: created_at -> created_ts (timestamp)
3. Derive event_date from  created_ts (timestamp)
4. Normalize event_type to lowercase
5. Extract nested identifiers:
        repo_id <- repo.id
        actor_login <- actor.login
##### Output columns must include:
```
event_id
event_type
created_ts
event_date
repo_id
actor_login
```
**_`Method 4`_**
**build_repo_scd2**(self, repo_changes_df: DataFrame) -> DataFrame
Build an SCD Type 2 dimension. Each change row represents a complete repository snapshot.
1. Parse: changed_at into timestamp.
2. For each: repo_id, sort changed_at ASC, Late-arriving rows must still appear in correct chronological order.
3. Compute:
valid_from = changed_at
valid_to = next row's changed_at for same repo_id, or NULL for the most recent row. 
is_current = valid_to IS NULL

##### Output columns:
```
repo_id
repo_name
visibility
default_branch
valid_from
valid_to
is_current
```
**_`Method 5`_**
**sessionize_actor_events**(self, flat_events_df: DataFrame) -> DataFrame
Group each actor's activity into work sessions.

1. Exclude rows where: actor_login IS NULL
2. Within an actor: Order by created_ts, event_id
3. A new session begins when: gap > 30 minutes (gap > 1800 seconds) . Gap of exactly 30 minutes belongs to the same session.
4. Assign: session_id(int): as a 1-based counter per actor.
5. Return all original columns plus: + session_id

**_`Method 6`_**
**enrich_with_repo_state**(sessionized_df, repo_scd2_df) -> DataFrame

1. Perform a Point-In-Time Join between events and scd2 dimensions. 
2. For each event, find the single SCD2 record satisfying:
like: repo_scd2.repo_id = event.repo_id AND valid_from <= created_ts AND (valid_to IS NULL OR created_ts < valid_to )
3. Append: repo_name, visibility, default_branch
4. If no matching SCD row is found: Keep the event. Populate appended columns as NULL. Use LEFT JOIN semantics.
5. Preserve all original event columns from flat_event_df exactly 

**_`Method 7`_**
**compute_session_summary**(self, enriched_sessionized_df: DataFrame) -> DataFrame

Create one summary row per: (actor_login, session_id)
actor_login(string)
session_id(int)
session_start(timestamp); the min created_ts in the session
session_end(timestamp): the max created_ts in the session
distinct_repos_touched(int): Count distinct: repo_id within the session.
events_in_session(int): Total event count in the session
primary_repo_at_session_start(string): enriched repo_name from the first event in the session. The first event is determined by: ORDER BY created_ts, event_id. If the first event did not match an SCD record: primary_repo_at_session_start = NULL

**_Expected Pipeline Flow_**
```
Python
def main()
job = PySparkJob()
events_df = job.read_events("data/events.jsonl")
repo_changes_df = job.read_repo_changes("data/repo_changes.jsonl")
flat_df = job.flatten_events(events_df)
scd2_df = job.build_repo_scd2(repo_changes_df)
sessionized_df = job.sessionize_actor_events(flat_df)
enriched_df = job.enrich_with_repo_state(
sessionized_df,
scd2_df
)
summary_df = job.compute_session_summary(
enriched_df
)
summary_df \
.orderBy("actor_login", "session_id") \
.show(truncate=False)
```







In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

class PySparkJob:

    def __init__(self, spark: SparkSession = None):
        self.spark = spark or SparkSession.builder.getOrCreate()

    # ------------------------------------------------------------------
    # Method 1: Read nested event data
    # ------------------------------------------------------------------
    def read_events(self, input_path: str) -> DataFrame:
        """
        Reads JSON Lines event data using an explicit schema.
        Nested fields are nullable so that missing fields do not cause
        schema-related failures.
        """
        event_schema = T.StructType([
            T.StructField("event_id", T.StringType(), True),
            T.StructField("event_type", T.StringType(), True),
            T.StructField("created_at", T.StringType(), True),
            T.StructField("repo",T.StructType([T.StructField("id", T.LongType(), True),
                                               T.StructField("name", T.StringType(), True)]),True),
            T.StructField("actor",T.StructType([T.StructField("id", T.LongType(), True),
                                                T.StructField("login", T.StringType(), True)]),True)])
        
        return (self.spark.read
                    .schema(event_schema)
                    .option("mode", "PERMISSIVE")
                    .json(input_path))
    # ------------------------------------------------------------------
    # Method 2: Read repository change data
    # ------------------------------------------------------------------
    def read_repo_changes(self, input_path: str) -> DataFrame:
        """
        Reads repository metadata change events using an explicit schema.
        """
        repo_change_schema = T.StructType([
            T.StructField("change_id", T.StringType(), True),
            T.StructField("repo_id", T.LongType(), True),
            T.StructField("changed_at", T.StringType(), True),
            T.StructField("repo_name", T.StringType(), True),
            T.StructField("visibility", T.StringType(), True),
            T.StructField("default_branch", T.StringType(), True)
        ])

        return (self.spark.read
                    .schema(repo_change_schema)
                    .option("mode", "PERMISSIVE")
                    .json(input_path))
    # ------------------------------------------------------------------
    # Method 3: Flatten nested event data
    # ------------------------------------------------------------------
    def flatten_events(self, events_df: DataFrame) -> DataFrame:
        """
        Converts nested events into an analysis-friendly structure.
        """
        created_ts = F.to_timestamp(F.col("created_at"))

        return events_df.select(
            F.col("event_id"),
            F.lower(F.col("event_type")).alias("event_type"),
            created_ts.alias("created_ts"),
            F.to_date(created_ts).alias("event_date"),
            F.col("repo.id").cast("long").alias("repo_id"),
            F.col("actor.login").alias("actor_login")
        )

    # ------------------------------------------------------------------
    # Method 4: Build repository SCD Type 2
    # ------------------------------------------------------------------
    def build_repo_scd2(self, repo_changes_df: DataFrame) -> DataFrame:
        """
        Builds SCD Type 2 history from complete repository snapshots.
        The source is append-only, but rows are reordered by changed_at
        so that late-arriving changes are placed in the proper timeline.
        """
        parsed_df = repo_changes_df.withColumn("changed_ts",F.to_timestamp(F.col("changed_at")))

        # change_id is included as a deterministic tie-breaker when two
        # changes for the same repository have the same timestamp.
        repo_window = (Window.partitionBy("repo_id")
                             .orderBy(F.col("changed_ts").asc(),F.col("change_id").asc()))

        scd2_df = (parsed_df
            .withColumn("valid_from", F.col("changed_ts"))
            .withColumn("valid_to",F.lead("changed_ts").over(repo_window))
            .withColumn("is_current",F.col("valid_to").isNull()))

        return scd2_df.select(
            F.col("repo_id"),
            F.col("repo_name"),
            F.col("visibility"),
            F.col("default_branch"),
            F.col("valid_from"),
            F.col("valid_to"),
            F.col("is_current")
        )
    # ------------------------------------------------------------------
    # Method 5: Sessionize actor events
    # ------------------------------------------------------------------
    def sessionize_actor_events(self, flat_events_df: DataFrame) -> DataFrame:
        """
        Assigns a 1-based session ID to each actor's events.
        A session boundary is created only when the gap is greater
        than 1,800 seconds. Therefore, a gap of exactly 30 minutes
        remains in the same session.
        """
        valid_actor_df = flat_events_df.filter(F.col("actor_login").isNotNull())

        actor_order_window = (Window.partitionBy("actor_login")
                              .orderBy(F.col("created_ts").asc(),
                                       F.col("event_id").asc()))

        actor_cumulative_window = (actor_order_window.rowsBetween(Window.unboundedPreceding,Window.currentRow))

        sessionized_df = (valid_actor_df
                            .withColumn("_previous_created_ts",F.lag("created_ts")
                                            .over(actor_order_window))
                            .withColumn("_gap_seconds",
                                            F.col("created_ts").cast("long")- 
                                            F.col("_previous_created_ts").cast("long"))
                            .withColumn("_new_session",
                                            F.when(F.col("_previous_created_ts").isNull(),F.lit(1))
                                             .when(F.col("_gap_seconds") > F.lit(1800),F.lit(1))
                                             .otherwise(F.lit(0)))
                            .withColumn("session_id",F.sum("_new_session")
                                            .over(actor_cumulative_window).cast("int"))
                            .drop("_previous_created_ts","_gap_seconds","_new_session"))

        return sessionized_df
    # ------------------------------------------------------------------
    # Method 6: Point-in-Time join
    # ------------------------------------------------------------------
    def enrich_with_repo_state(self,sessionized_df: DataFrame,repo_scd2_df: DataFrame) -> DataFrame:
        """
        Enriches each event with the repository metadata that was valid
        at the event timestamp.
        Validity interval:
            valid_from <= created_ts < valid_to
        For the current record, valid_to is NULL.
        """
        event_alias = "event"
        repo_alias = "repo_dim"

        events = sessionized_df.alias(event_alias)
        repo_dimension = repo_scd2_df.alias(repo_alias)

        join_condition = ((F.col(f"{event_alias}.repo_id")== F.col(f"{repo_alias}.repo_id"))
            & (F.col(f"{repo_alias}.valid_from") <= F.col(f"{event_alias}.created_ts"))
            & (F.col(f"{repo_alias}.valid_to").isNull() | 
                    (F.col(f"{event_alias}.created_ts") < F.col(f"{repo_alias}.valid_to"))))

        return (events.join(repo_dimension,join_condition,"left")
                      .select(F.col(f"{event_alias}.*"),
                              F.col(f"{repo_alias}.repo_name").alias("repo_name"),
                              F.col(f"{repo_alias}.visibility").alias("visibility"),
                              F.col(f"{repo_alias}.default_branch").alias("default_branch")))
    # ------------------------------------------------------------------
    # Method 7: Compute session summary
    # ------------------------------------------------------------------
    def compute_session_summary(self, enriched_sessionized_df: DataFrame) -> DataFrame:
        """
        Produces one row per actor session.
        The primary repository is taken from the chronologically first
        event based on created_ts and event_id.
        """

        first_event_window = (Window.partitionBy("actor_login", "session_id")
                                    .orderBy(F.col("created_ts").asc(),F.col("event_id").asc()))

        ranked_df = enriched_sessionized_df.withColumn("_event_position",
                                                        F.row_number().over(first_event_window))

        summary_df = (ranked_df.groupBy("actor_login","session_id")
                               .agg(F.min("created_ts").alias("session_start"),
                                    F.max("created_ts").alias("session_end"),
                                    F.countDistinct("repo_id").cast("int").alias("distinct_repos_touched"),
                                    F.count(F.lit(1)).cast("int").alias("events_in_session"),
                                    F.max(F.when(F.col("_event_position") == 1,F.col("repo_name")))
                                           .alias("primary_repo_at_session_start")))

        return summary_df.select(F.col("actor_login"),
                                 F.col("session_id").cast("int").alias("session_id"),
                                 F.col("session_start"),
                                 F.col("session_end"),
                                 F.col("distinct_repos_touched"),
                                 F.col("events_in_session"),
                                 F.col("primary_repo_at_session_start"))
        
def main():
    spark = (SparkSession.builder
        .appName("RepoSphereSessionizationPipeline")
        .getOrCreate())

    job = PySparkJob(spark)

#    print("Events df")
    events_df = job.read_events("/Volumes/workspace/default/landing_volume/events.json")
#    events_df.show(5)
#    print("Repo df")
    repo_changes_df = job.read_repo_changes("/Volumes/workspace/default/landing_volume/repo_changes.json")
#    repo_changes_df.show(5)

#    print("flat df")
    flat_df = job.flatten_events(events_df)
#    flat_df.show(5)
#    print("scd2 df")
    scd2_df = job.build_repo_scd2(repo_changes_df)
#    scd2_df.show(5)

#    print('sessioned_df')
    sessionized_df = job.sessionize_actor_events(flat_df)
#    sessionized_df.show(5)

#    print('enriched_df')
    enriched_df = job.enrich_with_repo_state(sessionized_df,scd2_df)
#    enriched_df.show(5)

    summary_df = job.compute_session_summary(enriched_df)
    (summary_df.orderBy("actor_login", "session_id").show(truncate=False))

if __name__ == "__main__":
    main()
    